# Tools

Tools are treated as functions that an LLM can call within an agent system.

In `smolagents`, tools can be defined in two ways:
1. **Using the `@tool` decorator** for simple function-based tools
2. **Creating a subclass of `Tool`** for more complex functionality

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

# Use this and `InferenceClientModel` for HuggingFace Inference API
HF_TOKEN = os.environ.get("HF_TOKEN")

# Use this for local LLM via Ollama (make sure to run `ollama serve` in the terminal first)
from smolagents import LiteLLMModel
model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
)

In [2]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")


from openinference.instrumentation.smolagents import SmolagentsInstrumentor
SmolagentsInstrumentor().instrument()

/home/haller/.conda/envs/hf-agents-course/lib/python3.14/site-packages/langfuse/api/core/pydantic_utilities.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.datetime_parse import parse_date as parse_date


Langfuse client is authenticated and ready!


## The @tool Decorator

The `@tool` decorator is the **recommended way to define simple tools**. Under the hood, smolagents will parse basic information about the function from Python. So if you name your function clearly and write a good docstring, it will be easier for the LLM to use.

Using this approach, we define a function with:

- **A clear and descriptive function name** that helps the LLM understand its purpose.
- **Type hints for both inputs and outputs** to ensure proper usage.
- **A detailed description**, including an `Args:` section where each argument is explicitly described. These descriptions provide valuable context for the LLM, so it's important to write them carefully.

Generating a tool that retrieves the highest-rated catering


In [3]:
from smolagents import CodeAgent, InferenceClientModel, tool

# Let's pretend we have a function that fetches the highest-rated catering services.
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


agent = CodeAgent(tools=[catering_service_tool], model=model)

# Run the agent to find the best catering service
result = agent.run(
    "Can you give me the name of the highest-rated catering service in Gotham City?"
)

print(result)   # Output: Gotham Catering Co.

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Can you give me the name of the highest-rated catering service in Gotham City?                                  │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen3.5:9b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  answer = catering_service_tool(query="Gotham City highest rated catering service")                               
  print(answer)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Catering Co.

Out: None

[Step 1: Duration 7.39 seconds| Input tokens: 2,193 | Output tokens: 145]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Gotham Catering Co.")                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Catering Co.

[Step 2: Duration 2.96 seconds| Input tokens: 4,548 | Output tokens: 258]

Gotham Catering Co.


## Defining a Tool as a Python Class


This approach involves creating a subclass of [`Tool`](https://huggingface.co/docs/smolagents/v1.8.1/en/reference/tools#smolagents.Tool).  For complex tools, we can implement a class instead of a Python function. The class wraps the function with metadata that helps the LLM understand how to use it effectively. In this class, we define:

- `name`: The tool's name.
- `description`: A description used to populate the agent's system prompt.
- `inputs`: A dictionary with keys `type` and `description`, providing information to help the Python interpreter process inputs.
- `output_type`: Specifies the expected output type.
- `forward`: The method containing the inference logic to execute.

In [4]:
from smolagents import Tool, CodeAgent, InferenceClientModel

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

# Instantiate the tool
party_theme_tool = SuperheroPartyThemeTool()
agent = CodeAgent(tools=[party_theme_tool], model=model)

# Run the agent to generate a party theme idea
result = agent.run(
    "What would be a good superhero party idea for a 'villain masquerade' theme?"
)

print(result)  # Output: "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What would be a good superhero party idea for a 'villain masquerade' theme?                                     │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen3.5:9b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  theme = superhero_party_theme_generator(category='villain masquerade')                                           
  print(theme)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 3.73 seconds| Input tokens: 2,229 | Output tokens: 201]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.")      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

[Step 2: Duration 2.82 seconds| Input tokens: 4,661 | Output tokens: 301]

Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.


In [5]:
# Uploading the tool to hf hub
party_theme_tool.push_to_hub("omri898/party_theme_tool")

README.md:   0%|          | 0.00/259 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/spaces/omri898/party_theme_tool/commit/91827e5504f4113642a929824e5b9553e2ee017c', commit_message='Upload tool', commit_description='', oid='91827e5504f4113642a929824e5b9553e2ee017c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/omri898/party_theme_tool', endpoint='https://huggingface.co', repo_type='space', repo_id='omri898/party_theme_tool'), pr_revision=None, pr_num=None)

## Importing a Tool from the Hub


In [6]:
from smolagents import load_tool, CodeAgent, InferenceClientModel

image_generation_tool = load_tool(
    "m-ric/text-to-image",
    trust_remote_code=True
)

agent = CodeAgent(
    tools=[image_generation_tool],
    model=model
)

agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")

tool.py:   0%|          | 0.00/635 [00:00<?, ?B/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.                 │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen3.5:9b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  image = image_generator("A luxurious superhero-themed party at Wayne Manor, featuring made-up superheroes in     
  elegant costumes, lavish decorations, crystal chandeliers, grand staircase, opulent rooms, champagne towers,     
  sophisticated guests, dramatic lighting, high-resolution, photorealistic")                                       
  final_answer(image)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'image = image_generator("A luxurious superhero-themed party at Wayne Manor, 
featuring made-up superheroes in elegant costumes, lavish decorations, crystal chandeliers, grand staircase, 
opulent rooms, champagne towers, sophisticated guests, dramatic lighting, high-resolution, photorealistic")' due 
to: HfHubHTTPError: 402 Client Error: Payment Required for url: 
https://router.huggingface.co/nscale/v1/images/generations (Request ID: 
Root=1-6a058e89-2682bc2f4f958e341b0954f6;f1d745ed-c927-49f4-9cf4-38e8d1d8540b)

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("I am unable to generate the image right now. The image_generator tool is currently unavailable     
  due to payment/credit restrictions. A luxurious superhero-themed party at Wayne Manor with made-up superheroes   
  would feature elegant costumes, opulent decorations, crystal chandeliers, and sophisticated guests celebrating   
  at Bruce Wayne's iconic estate.")                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: I am unable to generate the image right now. The image_generator tool is currently unavailable due to
payment/credit restrictions. A luxurious superhero-themed party at Wayne Manor with made-up superheroes would 
feature elegant costumes, opulent decorations, crystal chandeliers, and sophisticated guests celebrating at Bruce 
Wayne's iconic estate.

[Step 3: Duration 4.41 seconds| Input tokens: 7,836 | Output tokens: 1,029]

"I am unable to generate the image right now. The image_generator tool is currently unavailable due to payment/credit restrictions. A luxurious superhero-themed party at Wayne Manor with made-up superheroes would feature elegant costumes, opulent decorations, crystal chandeliers, and sophisticated guests celebrating at Bruce Wayne's iconic estate."

## Importing a Hugging Face Space as a Tool


This uses the HF API (paid)

In [4]:
from smolagents import CodeAgent, InferenceClientModel, Tool

image_generation_tool = Tool.from_space(
    "black-forest-labs/FLUX.1-schnell",
    name="image_generator",
    description="Generate an image from a prompt"
)

agent = CodeAgent(tools=[image_generation_tool], model=model)

agent.run(
    "Improve this prompt, then generate an image of it.",
    additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
)

Loaded as API: https://black-forest-labs-flux-1-schnell.hf.space


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Improve this prompt, then generate an image of it.                                                              │
│ You have been provided with these additional arguments, that you can access directly using the keys as          │
│ variables:                                                                                                      │
│ {'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}.      │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen3.5:9b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Original prompt                                                                                                
  original_prompt = 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'       
                                                                                                                   
  # Improved prompt with more detail                                                                               
  improved_prompt = "Elegant evening superhero-themed gala at Wayne Manor's grand hall, Alfred Pennyworth in       
  formal morning coat and bowtie standing center overseeing the luxurious celebration, decorated with              
  Batman-themed accents, elegant table settings with candles, dramatic cinematic lighting, guests in superhero     
  costumes, Gotham City backdrop, photorealistic, highly detailed, 8k, cinematic composition"                      
                                                                                                                   
  print(f"Improved prompt:\n{improved_prompt}")                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Improved prompt:
Elegant evening superhero-themed gala at Wayne Manor's grand hall, Alfred Pennyworth in formal morning coat and 
bowtie standing center overseeing the luxurious celebration, decorated with Batman-themed accents, elegant table 
settings with candles, dramatic cinematic lighting, guests in superhero costumes, Gotham City backdrop, 
photorealistic, highly detailed, 8k, cinematic composition

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Try with default/random parameters                                                                             
  prompt = "A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala"                
                                                                                                                   
  # Attempt to generate image with minimal parameters                                                              
  try:                                                                                                             
      image = image_generator(                                                                                     
          prompt=prompt                                                                                            
      )                                                                                                            
      final_answer(image)                                                                                          
  except Exception as e:                                                                                           
      print(f"Error: {e}")                                                                                         
      final_answer("Unable to generate image due to system error")                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Error: RuntimeError

Final answer: Unable to generate image due to system error

[Step 5: Duration 9.51 seconds| Input tokens: 15,186 | Output tokens: 1,586]

'Unable to generate image due to system error'

Defining a local image-generating tool

In [ ]:
from smolagents import tool, CodeAgent
from diffusers import DiffusionPipeline
import torch


pipe = DiffusionPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", dtype=torch.bfloat16, device_map="cuda")

prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
image = pipe(prompt).images[0]
print(type(image))
image.show()

In [4]:
from smolagents import tool, CodeAgent
from diffusers import DiffusionPipeline
import torch

@tool
def image_generator(prompt: str) -> "Image":
    """
    Generates an image from a text prompt using FLUX.1-schnell locally.

    Args:
        prompt: Detailed text description of the image to generate.
    """
    pipe = DiffusionPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", dtype=torch.bfloat16, device_map="cuda")
    image = pipe(prompt).images[0]
    return image

agent = CodeAgent(tools=[image_generator], model=model)
agent.run(
    "Improve this prompt, then generate an image of it.",
    additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
)

NameError: name 'Image' is not defined